In [1]:
%pip install --force-reinstall "numpy==2.2.6"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 80.9 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.1.3
    Uninstalling numpy-2.1.3:
      Successfully uninstalled numpy-2.1.3


In [1]:
raw_data = """
SUBJECT: DBMS
LEVEL: B.Tech CSE
YEARS: 2021-2025

[2021]
1. Explain the three-schema architecture of a DBMS.
2. Differentiate between primary keys and foreign keys.
3. Explain entity, attributes and relationships in an ER model.
4. Explain relational algebra operations.
5. Describe normalization and its objectives.

[2022]
1. Explain the relational model and integrity constraints.
2. Convert an ER diagram into relational tables.
3. Write SQL queries using joins and aggregate functions.
4. Explain functional dependencies.
5. Differentiate between 2NF, 3NF and BCNF.

[2023]
1. Explain relational algebra with examples.
2. Write SQL queries using nested queries.
3. Explain lossless decomposition and dependency preservation.
4. Explain transaction ACID properties.
5. Determine whether a transaction schedule is conflict-serializable.

[2024]
1. Explain different types of SQL joins.
2. Explain normalization using functional dependencies.
3. Describe transaction states and ACID properties.
4. Explain concurrency-control techniques.
5. Explain indexing and B+ trees.

[2025]
1. Explain relational algebra operators.
2. Write SQL queries involving joins and subqueries.
3. Explain functional dependencies and normalization.
4. Identify conflict-serializable schedules.
5. Explain transaction properties and recovery.


SUBJECT: DATA STRUCTURES
LEVEL: B.Tech CSE
YEARS: 2021-2025

[2021]
1. Explain arrays and linked lists.
2. Write algorithms for stack operations.
3. Convert an infix expression into postfix notation.
4. Explain binary-tree traversal techniques.
5. Compare linear search and binary search.

[2022]
1. Explain circular queues and priority queues.
2. Write algorithms for linked-list insertion and deletion.
3. Construct a binary search tree for a given sequence.
4. Explain sorting algorithms and their time complexities.
5. Explain hashing and collision resolution.

[2023]
1. Analyze the time complexity of common algorithms.
2. Explain recursion using the Tower of Hanoi problem.
3. Convert a postfix expression into its evaluated result.
4. Explain BFS and DFS graph traversals.
5. Explain AVL trees and balancing rotations.

[2024]
1. Derive the time complexity of merge sort.
2. Evaluate prefix and postfix expressions using stacks.
3. Write algorithms for stack and queue operations.
4. Explain hash functions with examples.
5. Construct a binary search tree and perform traversals.

[2025]
1. Analyze best-case, average-case and worst-case complexity.
2. Construct a binary search tree and find its traversals.
3. Convert an infix expression to postfix notation.
4. Explain circular-queue operations.
5. Explain threaded binary trees and graph traversal.
"""

print("Dataset text loaded.")

Dataset text loaded.


In [2]:
import re
import pandas as pd

records = []
subject = None
year = None

for line in raw_data.splitlines():
    line = line.strip()

    if line.startswith("SUBJECT:"):
        subject = line.split(":", 1)[1].strip().upper()

    elif re.fullmatch(r"\[\d{4}\]", line):
        year = int(line[1:-1])

    else:
        match = re.match(r"^\d+\.\s*(.+)$", line)

        if match and subject and year:
            records.append({
                "subject": subject,
                "year": year,
                "question": match.group(1),
                "source_type": "representative_practice"
            })

df = pd.DataFrame(records).drop_duplicates(
    subset=["subject", "year", "question"]
).reset_index(drop=True)

assert len(df) == 50, f"Expected 50 questions, found {len(df)}"

print("Dataset ready:", len(df), "questions")
display(df.groupby("subject").size().rename("Question count"))
display(df.head())

Dataset ready: 50 questions


,Question count
subject,
DATA STRUCTURES,25
DBMS,25


,subject,year,question,source_type
0,DBMS,2021,Explain the three-schema architecture of a DBMS.,representative_practice
1,DBMS,2021,Differentiate between primary keys and foreign...,representative_practice
2,DBMS,2021,"Explain entity, attributes and relationships i...",representative_practice
3,DBMS,2021,Explain relational algebra operations.,representative_practice
4,DBMS,2021,Describe normalization and its objectives.,representative_practice


In [5]:
%pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 76.9 MB/s eta 0:00:00


In [6]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

question_texts = df["question"].tolist()

embeddings = embedding_model.encode(
    question_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")

# Map vector positions back to DataFrame row positions.
vector_index = faiss.IndexFlatIP(embeddings.shape[1])
vector_index.add(embeddings)

print("RAG index ready:", vector_index.ntotal, "questions")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

RAG index ready: 50 questions


In [7]:
TOPIC_KEYWORDS = {
    "Database Architecture and Keys": [
        "three-schema", "primary keys", "foreign keys",
        "integrity constraints", "relational model"
    ],
    "ER Model": [
        "er model", "er diagram", "entity, attributes",
        "entity and relationships"
    ],
    "Relational Algebra": [
        "relational algebra"
    ],
    "SQL and Joins": [
        "sql queries", "sql", "joins", "subqueries",
        "nested queries", "aggregate functions"
    ],
    "Normalization": [
        "normalization", "2nf", "3nf", "bcnf"
    ],
    "Functional Dependencies": [
        "functional dependenc"
    ],
    "Transactions and Concurrency": [
        "transaction", "serializab", "concurrency control",
        "acid properties", "recovery"
    ],
    "Indexing": [
        "indexing", "b+ trees"
    ],
    "Arrays and Linked Lists": [
        "arrays", "linked list"
    ],
    "Stacks and Queues": [
        "stack", "queue", "infix", "postfix", "prefix"
    ],
    "Trees": [
        "binary tree", "binary search tree", "avl",
        "threaded binary"
    ],
    "Sorting and Complexity": [
        "sorting", "time complexity", "space complexity",
        "best-case", "merge sort"
    ],
    "Hashing": [
        "hashing", "hash functions", "collision resolution"
    ],
    "Searching": [
        "linear search", "binary search"
    ],
    "Graphs": [
        "bfs", "dfs", "graph traversal"
    ],
    "Recursion": [
        "recursion", "tower of hanoi"
    ]
}


def get_topic_report(subject):
    subset = df[df["subject"] == subject.upper()]
    results = []

    for topic, keywords in TOPIC_KEYWORDS.items():
        matched = subset[
            subset["question"].apply(
                lambda question: any(
                    keyword in question.lower()
                    for keyword in keywords
                )
            )
        ]

        if matched.empty:
            continue

        results.append({
            "Topic": topic,
            "Question count": len(matched),
            "Years present": matched["year"].nunique(),
            "Years": ", ".join(
                map(str, sorted(matched["year"].unique()))
            ),
            "Matching questions": matched["question"].tolist()
        })

    results.sort(
        key=lambda row: (
            row["Years present"],
            row["Question count"]
        ),
        reverse=True
    )

    return results


for item in get_topic_report("DBMS")[:5]:
    print(item["Topic"], "-", item["Question count"], "matching questions")

SQL and Joins - 4 matching questions
Normalization - 4 matching questions
Transactions and Concurrency - 5 matching questions
Relational Algebra - 3 matching questions
Functional Dependencies - 3 matching questions


In [8]:
PRACTICE_QUESTIONS = {
    "DBMS": [
        ("Normalization", "Explain 1NF, 2NF, 3NF and BCNF."),
        ("SQL and Joins", "Write SQL queries using joins and aggregate functions."),
        ("Relational Algebra", "Solve relational algebra queries using selection and projection."),
        ("Functional Dependencies", "Find candidate keys using functional dependencies."),
        ("Transactions and Concurrency", "Determine whether a transaction schedule is conflict-serializable."),
        ("Indexing", "Explain B-tree and B+ tree indexing."),
        ("ER Model", "Convert an ER diagram into relational tables."),
    ],
    "DATA STRUCTURES": [
        ("Arrays and Linked Lists", "Write algorithms for linked-list insertion and deletion."),
        ("Stacks and Queues", "Convert an infix expression into postfix notation."),
        ("Trees", "Construct a binary search tree and perform its traversals."),
        ("Sorting and Complexity", "Analyze the time complexity of merge sort."),
        ("Hashing", "Explain collision-resolution techniques in hashing."),
        ("Graphs", "Perform BFS and DFS traversals on a graph."),
        ("Recursion", "Solve the Tower of Hanoi problem using recursion."),
    ]
}


def retrieve_questions(query, subject, k=5):
    subset_indices = df.index[
        df["subject"] == subject.upper()
    ].to_numpy()

    if len(subset_indices) == 0:
        return []

    query_vector = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = vector_index.search(
        query_vector, vector_index.ntotal
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):
        if idx < 0:
            continue

        if idx not in subset_indices:
            continue

        row = df.iloc[int(idx)]

        results.append({
            "year": int(row["year"]),
            "question": row["question"],
            "score": round(float(score), 3)
        })

        if len(results) >= k:
            break

    return results


def answer_bot(message, subject):
    query = message.strip()
    q = query.lower()
    subject = subject.upper()

    if not query:
        return "Enter a question to get started."

    # Topic frequency
    if any(term in q for term in [
        "important topic", "important topics",
        "repeat most", "repeated topic", "repeated topics",
        "most repeated", "frequency", "frequently asked",
        "topic count", "topic counts"
    ]):
        report = get_topic_report(subject)

        if not report:
            return "No matching topics found."

        lines = [
            "## Important Topics",
            f"**Subject:** {subject}",
            "",
            "| Rank | Topic | Matching questions | Years present |",
            "|---:|---|---:|---:|"
        ]

        for rank, item in enumerate(report, 1):
            lines.append(
                f"| {rank} | {item['Topic']} | "
                f"{item['Question count']} | "
                f"{item['Years present']} |"
            )

        lines.extend([
            "",
            "### Questions grouped by topic"
        ])

        for item in report[:5]:
            lines.append(
                f"\n**{item['Topic']} — "
                f"{item['Question count']} matching questions**"
            )

            for question in item["Matching questions"]:
                lines.append(f"- {question}")

        lines.append(
            "\n*Counts are based on the supplied practice dataset, "
            "not verified historical examination statistics.*"
        )

        return "\n".join(lines)

    # Five practice questions
    if any(term in q for term in [
        "likely question", "likely questions",
        "practice question", "practice questions",
        "give me 5 questions", "generate questions"
    ]):
        report = get_topic_report(subject)
        ranked_topics = [item["Topic"] for item in report]

        candidates = PRACTICE_QUESTIONS.get(subject, [])
        selected = []

        for topic in ranked_topics:
            for candidate_topic, question in candidates:
                if candidate_topic == topic and question not in selected:
                    selected.append(question)

                if len(selected) == 5:
                    break

            if len(selected) == 5:
                break

        return "## 5 Practice Questions\n\n" + "\n".join(
            f"{i}. {question}"
            for i, question in enumerate(selected, 1)
        ) + "\n\n*Practice suggestions, not guaranteed exam predictions.*"

    # Semantic retrieval for other queries
    results = retrieve_questions(query, subject, k=5)

    if not results:
        return "No questions found for this subject."

    lines = ["## Related Questions"]

    for result in results:
        lines.append(
            f"- **[{result['year']}]** {result['question']} "
            f"*(similarity: {result['score']:.2f})*"
        )

    lines.append(
        "\n*Similarity is a retrieval score, not a repetition count.*"
    )

    return "\n".join(lines)


print(answer_bot("Which DBMS topics repeat most?", "DBMS"))

## Important Topics
**Subject:** DBMS

| Rank | Topic | Matching questions | Years present |
|---:|---|---:|---:|
| 1 | SQL and Joins | 4 | 4 |
| 2 | Normalization | 4 | 4 |
| 3 | Transactions and Concurrency | 5 | 3 |
| 4 | Relational Algebra | 3 | 3 |
| 5 | Functional Dependencies | 3 | 3 |
| 6 | Database Architecture and Keys | 3 | 2 |
| 7 | ER Model | 2 | 2 |
| 8 | Indexing | 1 | 1 |

### Questions grouped by topic

**SQL and Joins — 4 matching questions**
- Write SQL queries using joins and aggregate functions.
- Write SQL queries using nested queries.
- Explain different types of SQL joins.
- Write SQL queries involving joins and subqueries.

**Normalization — 4 matching questions**
- Describe normalization and its objectives.
- Differentiate between 2NF, 3NF and BCNF.
- Explain normalization using functional dependencies.
- Explain functional dependencies and normalization.

**Transactions and Concurrency — 5 matching questions**
- Explain transaction ACID properties.
- Determin

In [ ]:
import gradio as gr

CUSTOM_CSS = """
.gradio-container {
    max-width: 1150px !important;
    margin: auto !important;
}
#hero {
    text-align: center;
    padding: 18px 8px;
}
#hero h1 {
    font-size: 30px;
}
"""

def dashboard(subject):
    report = get_topic_report(subject)

    table = pd.DataFrame([
        {
            "Rank": i,
            "Topic": item["Topic"],
            "Matching Questions": item["Count"],
            "Years Present": item["Years Present"],
            "Years": item["Years"]
        }
        for i, item in enumerate(report, 1)
    ])

    summary = (
        f"**{subject}** · "
        f"{len(df[df['subject'] == subject])} questions · "
        f"{len(report)} topic groups"
    )

    return table, summary


def grouped_questions(subject, selected_topic):
    report = get_topic_report(subject)

    if selected_topic != "All topics":
        report = [
            item for item in report
            if item["Topic"] == selected_topic
        ]

    output = []

    for item in report:
        output.append(
            f"### {item['Topic']} — {item['Count']} matching questions"
        )
        output.append(f"**Years:** {item['Years']}")

        for question in item["Questions"]:
            output.append(f"- {question}")

        output.append("")

    if not output:
        return "No matching topic found for this subject."

    output.append(
        "*These counts represent topic matches in the sample dataset.*"
    )

    return "\n".join(output)


def chat_response(message, history, subject):
    if not message or not message.strip():
        return history, ""

    history = history or []
    response = answer_bot(message, subject)

    history = history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": response}
    ]

    return history, ""


with gr.Blocks(title="AI Exam Topic Analyzer") as demo:

    gr.Markdown(
        """
        <div id="hero">
        <h1>📚 AI Exam Topic Analyzer</h1>
        <p>Find important topics, repeated question patterns, and practice questions.</p>
        </div>
        """
    )

    gr.Markdown(
        "> **Dataset notice:** This application uses representative "
        "practice questions, not verified historical exam papers."
    )

    subject = gr.Dropdown(
        choices=["DBMS", "DATA STRUCTURES"],
        value="DBMS",
        label="Select Subject"
    )

    with gr.Tab("📊 Important Topics"):
        summary = gr.Markdown()
        topic_table = gr.Dataframe(
            label="Topic Frequency Ranking",
            interactive=False
        )

        refresh = gr.Button(
            "Refresh Dashboard",
            variant="primary"
        )

        refresh.click(
            fn=dashboard,
            inputs=subject,
            outputs=[topic_table, summary]
        )

        subject.change(
            fn=dashboard,
            inputs=subject,
            outputs=[topic_table, summary]
        )

    with gr.Tab("🔁 Repeated Questions"):
        topic_options = ["All topics"] + list(TOPIC_KEYWORDS.keys())

        selected_topic = gr.Dropdown(
            choices=topic_options,
            value="All topics",
            label="Filter by topic"
        )

        show_questions = gr.Button(
            "Show Questions",
            variant="primary"
        )

        questions_output = gr.Markdown()

        show_questions.click(
            fn=grouped_questions,
            inputs=[subject, selected_topic],
            outputs=questions_output
        )

    with gr.Tab("🎯 Practice Questions"):
        generate = gr.Button(
            "Generate 5 Questions",
            variant="primary"
        )

        practice_output = gr.Dataframe(
            label="Practice Questions",
            interactive=False
        )

        generate.click(
            fn=lambda s: pd.DataFrame(
                generate_practice_questions(s, 5)
            ),
            inputs=subject,
            outputs=practice_output
        )

    with gr.Tab("💬 Ask the Bot"):
        gr.Markdown(
            "Ask which topics repeat most, request five practice "
            "questions, or search for a topic."
        )

        chatbot = gr.Chatbot(
            label="Exam Topic Assistant",
            height=420
        )

        with gr.Row():
            message = gr.Textbox(
                label="Your question",
                placeholder="Which DBMS topics repeat most?",
                scale=5
            )

            send = gr.Button("Ask", variant="primary")

        clear = gr.Button("Clear Chat")

        send.click(
            fn=chat_response,
            inputs=[message, chatbot, subject],
            outputs=[chatbot, message]
        )

        message.submit(
            fn=chat_response,
            inputs=[message, chatbot, subject],
            outputs=[chatbot, message]
        )

        clear.click(
            fn=lambda: ([], ""),
            inputs=None,
            outputs=[chatbot, message]
        )

    demo.load(
        fn=dashboard,
        inputs=subject,
        outputs=[topic_table, summary]
    )

demo.launch(
    share=True,
    debug=True,
    theme=gr.themes.Soft(),
    css=CUSTOM_CSS
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://17942cd899dd452fee.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/gradio/queueing.py", line 848, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/route_utils.py", line 417, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<12 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/blocks.py", line 2698, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<9 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/blocks.py", line 1963, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        fn, *processed_input, limiter=self.limiter
        ^^^^^^^^^^^^^^